# Train YOLOv8n trên Kaggle

Khác với bản Colab:
- Thư mục chính là **/kaggle/working** (không phải /content)
- **Không cần mount Drive** — lấy kết quả bằng nút *Output → Download* bên phải
- Kaggle miễn phí có **30 giờ GPU/tuần** và 1 session chạy liên tục tới ~9-12h → ít bị ngắt giữa chừng hơn Colab

**Bước đầu tiên:** mở panel **Settings ⚙** bên phải → *Accelerator*: chọn **GPU T4 x2** (hoặc P100) → bật *Internet* ON (để pip + tải dataset) → *Save*.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("GPU available:", torch.cuda.is_available())

## 1. Tải dataset từ Roboflow

Dataset được export sang YOLOv8, kèm data.yaml (chứa tên class). Cần *Internet* bật trong Settings.

In [ ]:
import os
from roboflow import Roboflow

# ===== THÔNG TIN ROBOFLOW CỦA BẠN =====
# Đọc key từ biến môi trường (không hardcode để khỏi lộ trên GitHub)
ROBOFLOW_API_KEY  = os.environ.get("ROBOFLOW_API_KEY", "")
if not ROBOFLOW_API_KEY:
    raise ValueError("Thiếu biến môi trường ROBOFLOW_API_KEY. Đặt nó trước khi chạy.")
WORKSPACE         = "trantungbach26-gmail-com"
PROJECT_NAME      = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
version = project.version(PROJECT_VERSION)
version.download("yolov8")

In [ ]:
# Tự tìm thư mục dataset vừa tải, và đổi đường dẫn sang thư mục chuẩn tổ chức
import os, glob, shutil, yaml

candidates = glob.glob("/kaggle/working/*/data.yaml")
if candidates:
    DATASET_PATH = os.path.dirname(candidates[0])
else:
    DATASET_PATH = "/kaggle/working/citrus-disease-detection-1"

print("DATASET_PATH =", DATASET_PATH)

with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    cfg = yaml.safe_load(f)
print("Số class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load model & train

tự tải yolov8n.pt lần đầu (cần Internet).

In [ ]:
model = YOLO("yolov8n.pt")  # tự tải pretrained
print("Đã load yolov8n.pt")

In [ ]:
# ===== Cấu hình train (best practice) =====
EPOCHS = 50       # yolov8n đủ hội tụ; Kaggle ít bị cut nên không cần thấp quá
IMGSZ = 640       # khớp với export & kmodel
BATCH = 16
PATIENCE = 15

RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n/weights"
OUT_DIR     = "/kaggle/working/drone_yolo"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== BACKUP mỗi epoch: copy best.pt vào thư mục Output để tải được bất kỳ lúc nào =====
from ultralytics.utils import callbacks

def _backup(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))
        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}", flush=True)
    except Exception as e:
        print("  [backup fail]", e, flush=True)

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

print(f"Train yolov8n: epochs={EPOCHS}, imgsz={IMGSZ}, patience={PATIENCE}")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    cache=True,
    workers=2,
    cos_lr=True,
    project="/kaggle/working/runs",
    name="drone_yolov8n",
)

## 3. Đánh giá kết quả

In [ ]:
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
import glob

test_imgs = (glob.glob(f"{DATASET_PATH}/test/images/*.*")
             or glob.glob(f"{DATASET_PATH}/valid/images/*.*")
             or glob.glob(f"{DATASET_PATH}/val/images/*.*"))
print("Tìm thấy", len(test_imgs), "ảnh")

if test_imgs:
    model.predict(source=test_imgs[:4], conf=0.25, save=True, project="/kaggle/working/predict")
    saved = glob.glob("/kaggle/working/predict/**/*.jpg", recursive=True)
    if saved:
        from IPython.display import Image
        print("Ảnh đã lưu:", saved[0])
        display(Image(filename=saved[0]))
    else:
        print("Không tìm thấy ảnh đã lưu (bỏ qua preview, không crash)")
else:
    print("Không có ảnh test để preview")

## 4. Export 2 phiên bản

- **best.pt** → chạy realtime trên **laptop** (realtime_cam.py)
- **best.onnx** → qua nncase → **best.kmodel** chạy trên **drone K230**

In [ ]:
# Export ONNX dành cho K230. (best.pt cho laptop đã có sẵn trong thư mục weights.)
best = YOLO(f"{RESULTS_DIR}/best.pt")
best.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)

# Liệt kê các phiên bản đã có
print("\nĐã tạo/xuất các phiên bản:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if f in ("best.pt", "best.onnx"):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1e6
        print(f"  - {f}  ({size:.1f} MB)")
print("\nThư mục:", RESULTS_DIR)

In [ ]:
# Sao chép mọi phiên bản vào thư mục Output để tải về máy
import glob as _g

for f in _g.glob(RESULTS_DIR + "/best.*"):
    shutil.copy(f, os.path.join(OUT_DIR, os.path.basename(f)))
    print("Đã copy output:", os.path.basename(f))

print("\n>>> Tải kết quả: panel bên phải tab 'Output' -> biểu tượng tải xuống (Download all).")

## 5. Tùy chọn: convert ONNX → kmodel ngay trên Kaggle (Linux)

### Tạm dừng — xem side-note:

Để chắc chắn dùng đúng bản nncase của tài liệu CanMV (2.10.0), nhiều máy gặp lỗi phiên bản khi `pip install nncase` kéo bản mới. Cách an toàn nhất:

1. Tải **best.onnx** về máy Windows
2. Chạy `convert_to_kmodel.py` trên máy (đã có trong project) — đúng bản nncase 2.10.0 + nncase_kpu 2.10.0 (win_amd64 whl)

Nếu bạn muốn convert trên Kaggle, cài đúng bản:
```
pip install nncase==2.10.0
curl -L -O https://github.com/kendryte/nncase/releases/download/v2.10.0/nncase_kpu-2.10.0-py2.py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
pip install that.whl
```